# Práctica en Clase Spark SQL.

Respetado estudiante en este Notebook vamos a practicar la ejecución de consultas SQL sobre datos en SPARK usando el módulo SPARK - SQL.

In [ ]:
#Creamos la sección de Spark. 
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MiAplicacion").master("local[*]").config("spark.hadoop.fs.defaultFS","file:///").getOrCreate()

print(spark.version)

In [ ]:
#Leer los datos del csv clientes_1000
clientes_df = spark.read.csv(
    "./clientes_1000.csv",
    header=True,
    inferSchema=True
)

In [ ]:
#Revisar datos del df
clientes_df.show()

clientes_df.count()

In [ ]:
#Agregar nombre temporal de vista a Spark para hacer consultas sobre los datos
clientes_df.createOrReplaceTempView("clientes")

In [ ]:
#Primera Consulta SQL desde Spark.
spark.sql("""
    
""").show()

In [ ]:
#Segunda Consulta SQL desde Spark.
spark.sql("""
    
""").show()

In [ ]:
#Consulta SQL elaborada
spark.sql("""
    
""").show()

In [ ]:
# Agrupación + Agregación (Count)
spark.sql("""
    
""").show()

In [ ]:
#Calcular edad promedio de clientes por departamento
spark.sql("""
    
""").show()

In [ ]:
#Leer los datos del csv productos_1000
productos_df = spark.read.csv(
    "./productos_1000.csv",
    header=True,
    inferSchema=True
)

In [ ]:
productos_df.show()

In [ ]:
#Agregar nombre temporal de vista a Spark para hacer consultas sobre los datos
productos_df.createOrReplaceTempView("productos")

In [ ]:
#Importar funciones de Spark
from pyspark.sql import functions as F

In [ ]:
#Hacer una agrupación usando la API Dataframe de Spark.
resultado = (
    productos_df.groupBy("categoria")
      .agg(
          F.count("*").alias("cantidad_productos"),
          F.round(
              F.avg(
                  F.col("precio_unitario")
              ),2
          ).alias("precio_unitario_promedio")
      )
      .orderBy(F.desc("cantidad_productos"))
)

resultado.show()

In [ ]:
#Hacer la misma consulta pero desde SQL
spark.sql("""
    SELECT
        categoria,
        COUNT(*) as cantidad_productos,
        ROUND(AVG(precio_unitario),2) AS precio_unitario_promedio
    FROM productos
    GROUP BY categoria
    ORDER BY cantidad_productos DESC;
""").show()

In [ ]:
#Guardando el resultado de la API de SPARK.
resultado.write \
    .mode("overwrite") \
    .parquet("salidas/clientes_por_departamento")

In [ ]:
#Leer un archivo parquet
df_parquet = spark.read.parquet(
    "salidas/clientes_por_departamento"
)

df_parquet.show()
df_parquet.printSchema()

In [ ]:
#Guardar la consulta desde SQL (esto da como salida un DataFrame)
spark.sql("""
    CREATE TABLE IF NOT EXISTS resumen_productos
    USING PARQUET
    AS
    SELECT
        categoria,
        COUNT(*) as cantidad_productos,
        ROUND(AVG(precio_unitario),2) AS precio_unitario_promedio
    FROM productos
    GROUP BY categoria
    ORDER BY cantidad_productos DESC;
""")

In [ ]:
#Y podemos consultar esa tabla así:
spark.sql("""
    SELECT *
    FROM resumen_productos
    ORDER BY cantidad_productos DESC;
""").show()

In [ ]:
# Detener la sesión actual
spark.stop()